# P002 指派问题 · PySCIPOpt 建模与评测演示

**场景**：将 $n$ 名作业人员（或机台）一对一指派给 $n$ 项独立生产任务，在双向唯一匹配约束下最小化总作业成本。本案例演示标准平衡指派模型的 PySCIPOpt 实现与解矩阵提取。

练习用，不计正式成绩。

## 1. 混合整数线性规划模型

- 二元变量 $x_{ij} \in \{0, 1\}$ 表示人员 $i$ 是否承担任务 $j$；
- 目标函数：最小化总指派成本：

$$\min \sum_{i=0}^{n-1} \sum_{j=0}^{n-1} c_{ij} x_{ij}$$

- 行约束（每人恰分派一项任务）：$\sum_{j=0}^{n-1} x_{ij} = 1$，$\forall i$；
- 列约束（每项任务恰由一人承担）：$\sum_{i=0}^{n-1} x_{ij} = 1$，$\forall j$。

由于指派约束矩阵具备**全幺模性**，即使将变量改为连续非负，极点最优解也保持整数性。

In [ ]:
from pyscipopt import Model, quicksum
import numpy as np
print("PySCIPOpt 与 NumPy 加载成功！")

## 2. 接口契约：solve(instance) -> dict

输入数据规范：
- `n`：人数与任务数；
- `costs`：$n \times n$ 费用矩阵，`costs[i][j]` 为人员 $i$ 完成任务 $j$ 的成本。

返回值规范：
- `status`：求解状态（如 `"optimal"`）；
- `objective`：`float` 最小总成本；
- `solution`：$n \times n$ 二维矩阵，每行每列恰有一个 `1.0`，其余为 `0.0`。

In [ ]:
def solve(instance: dict) -> dict:
    n = instance["n"]
    costs = instance["costs"]

    model = Model("assignment")
    model.hideOutput(True)

    # x[i, j] = 1 表示指派人员 i 执行任务 j
    x = {}
    for i in range(n):
        for j in range(n):
            x[i, j] = model.addVar(vtype="B", name=f"x_{i}_{j}")

    # 目标函数：最小化总费用
    model.setObjective(quicksum(costs[i][j] * x[i, j] for i in range(n) for j in range(n)), "minimize")

    # 每个人恰好分配一个任务
    for i in range(n):
        model.addCons(quicksum(x[i, j] for j in range(n)) == 1)

    # 每项任务恰好分配给一个人
    for j in range(n):
        model.addCons(quicksum(x[i, j] for i in range(n)) == 1)

    model.optimize()

    sol = [[float(round(model.getVal(x[i, j]))) for j in range(n)] for i in range(n)]
    return {
        "status": "optimal" if model.getStatus() == "optimal" else str(model.getStatus()),
        "objective": float(model.getObjVal()),
        "solution": sol,
    }

In [ ]:
# 测试用例验证
test_instance = {
    "n": 3,
    "costs": [
        [9, 2, 7],
        [6, 4, 3],
        [5, 8, 1]
    ]
}
res = solve(test_instance)
print("指派求解结果:", res)

## 3. 可以接着改什么

1. 将矩阵中某格成本改为极大值（模拟人员不具备该技能），观察匹配方案如何重新分配；
2. 对费用矩阵任意一行同加固定常数，验证最优匹配组合不变而目标值同步增加该常数；
3. 准备提交 `P002` 正式作业时，可在工作台切换「分屏对照」对照题面与模板完成编码。